# On-disk function caching with `disk_cache`

`xpectral.utils.cache.disk_cache` memoizes a function's return values **on disk**,
keyed by its arguments. Unlike `functools.lru_cache`, the results persist across
processes and interpreter restarts, which is ideal for slow, repeatable work such
as API data pulls.

Each decorated function gets its own cache directory (no shared global instance),
backed by [`diskcache`](https://grantjenks.com/docs/diskcache/), which handles the
arg-hashing, atomic writes, TTL, size limits, and LRU eviction for us. Values are
pickled, so the function must take picklable/hashable arguments and return
picklable objects.

In [1]:
import tempfile
import time

from xpectral.utils.cache import disk_cache

# Keep this example self-contained: cache into a throwaway temp directory
# instead of the default ~/.cache/xpectral.
CACHE_DIR = tempfile.mkdtemp(prefix="xpectral-cache-demo-")
CACHE_DIR

'/var/folders/fl/458rlssj6bx0dl2smvzhsyvr0000gn/T/xpectral-cache-demo-kpujm83x'

## A slow function

We simulate expensive work with `time.sleep` and count how many times the body
actually runs, so we can prove the cache is doing its job.

In [2]:
calls = {"n": 0}


@disk_cache("demo", cache_dir=CACHE_DIR)
def slow_add(a, b):
    calls["n"] += 1
    time.sleep(0.5)  # pretend this is a network / heavy compute call
    return a + b

## Miss vs. hit

The first call runs the body (a *miss*); the second call with identical arguments
is served from disk (a *hit*) and skips the work entirely.

In [3]:
t0 = time.perf_counter()
r1 = slow_add(2, 3)  # miss
t1 = time.perf_counter()
r2 = slow_add(2, 3)  # hit
t2 = time.perf_counter()

print(f"result: {r1} -> {r2}")
print(f"miss took {t1 - t0:.3f}s")
print(f"hit  took {t2 - t1:.4f}s")
print(f"function body ran {calls['n']} time(s)")

assert r1 == r2
assert calls["n"] == 1  # the body ran only for the miss

result: 5 -> 5
miss took 0.505s
hit  took 0.0002s
function body ran 1 time(s)


## Expiring stale entries (TTL)

Pass `expire=<seconds>` to invalidate entries after a time-to-live. This matters
for data that goes stale (e.g. recent market bars). Here entries live for 1 second.

In [4]:
ttl_calls = {"n": 0}


@disk_cache("demo_ttl", expire=1, cache_dir=CACHE_DIR)
def slow_double(x):
    ttl_calls["n"] += 1
    return x * 2


slow_double(21)  # miss -> body runs
slow_double(21)  # hit  -> cached
print(f"after two quick calls: body ran {ttl_calls['n']} time(s)")

time.sleep(1.1)  # let the entry expire
slow_double(21)  # miss again -> body re-runs
print(f"after TTL expiry:      body ran {ttl_calls['n']} time(s)")

assert ttl_calls["n"] == 2

after two quick calls: body ran 1 time(s)


after TTL expiry:      body ran 2 time(s)


## Inspecting and clearing

The wrapped function exposes its backing store as `func.cache` (a
`diskcache.Cache`), so you can check how many entries it holds or clear them.

In [5]:
print(f"entries before clear: {len(slow_add.cache)}")

slow_add.cache.clear()
print(f"entries after clear:  {len(slow_add.cache)}")

# With the cache emptied, the next call is a miss again and the body re-runs.
before = calls["n"]
slow_add(2, 3)
print(f"body ran again after clear: {calls['n'] == before + 1}")

assert len(slow_add.cache) == 1
assert calls["n"] == before + 1

entries before clear: 1
entries after clear:  0


body ran again after clear: True


## Applying it to `Massive.get_aggregate_bars`

Caching the real data fetch takes two small precautions:

1. **Don't key on `self`.** `get_aggregate_bars` is a bound method, and its
   `Massive` instance holds a live `RESTClient` that is not a stable cache key.
   Cache a plain function of the *inputs* (tickers, timespan, date range, ...) and
   call it from the method — don't decorate the method directly.
2. **Cache data, not a `LazyFrame`.** The method returns `df.lazy()`; a `LazyFrame`
   is a deferred query plan and is fragile to pickle across polars versions. Cache
   the materialized `pl.DataFrame` and call `.lazy()` at the boundary.

The cell below is **illustrative** (it needs a `MASSIVE_API_KEY`), so it is guarded
and not run as part of this example.

In [6]:
RUN_LIVE = False  # set True with a MASSIVE_API_KEY in the environment

if RUN_LIVE:
    import polars as pl

    from massive import RESTClient
    from xpectral.utils.cache import disk_cache

    # A module-level, self-free fetch keyed only on the real inputs. Returns a
    # materialized DataFrame so the cached value is plain data, not a query plan.
    @disk_cache("get_aggregate_bars", expire=60 * 60 * 24)  # 1 day TTL
    def _fetch_bars(tickers, multiplier, timespan, from_, to):
        client = RESTClient()
        rows = []
        for ticker in tickers:
            for agg in client.list_aggs(
                ticker=ticker,
                multiplier=multiplier,
                timespan=timespan,
                from_=from_,
                to=to,
            ):
                rows.append(agg.__dict__ | {"ticker": ticker})
        return pl.DataFrame(rows)

    bars = _fetch_bars(["AAPL"], 1, "day", "2024-01-01", "2024-01-31").lazy()
    print(bars.collect())